# Circuit Discovery: Automated Circuit Finding

This notebook demonstrates IRTK's `circuit_discovery` module:

1. **Edge attribution** – importance of each computational edge
2. **Iterative pruning** – ACDC-style minimal circuit extraction
3. **Subnetwork probing** – random subset testing
4. **Path attribution** – through-layer path importance
5. **Full discovery** – end-to-end circuit finding pipeline

## Why This Matters

Automated circuit discovery (ACDC-style) uses iterative edge pruning and subnetwork probing to find minimal circuits for specific behaviors. This scales beyond manual circuit analysis by systematically testing which connections between components are necessary.

**Key references:**
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.circuit_discovery import (
    edge_attribution_matrix, iterative_circuit_pruning,
    subnetwork_probing, path_attribution, discover_circuit,
)

In [ ]:
cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
              if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 1, 2, 3, 4, 5])
def metric_fn(logits): return float(logits[-1, 0])

In [ ]:
edges = edge_attribution_matrix(model, tokens, metric_fn)
print(f"Edge matrix shape: {edges['edge_scores'].shape}")
print(f"Top edges: {edges['top_edges'][:3]}")
print(f"Sparsity: {edges['sparsity']:.2f}")

In [ ]:
circuit = iterative_circuit_pruning(model, tokens, metric_fn)
print(f"Circuit size: {circuit['circuit_size']}")
print(f"Components: {circuit['circuit_components']}")
print(f"Compression: {circuit['compression_ratio']:.2%}")

In [ ]:
probe = subnetwork_probing(model, [tokens], metric_fn, n_random_subsets=20)
print(f"Mean subset metric: {probe['mean_subset_metric']:.4f}")
print(f"Best subset: {probe['best_subset_metric']:.4f}")
print(f"Critical components: {probe['critical_components']}")

In [ ]:
full = discover_circuit(model, [tokens], metric_fn)
print(f"Nodes ({full['circuit_size']}): {full['nodes']}")
print(f"Edges: {full['edges']}")
print(f"Importance: {full['node_importance']}")